PRIMEIRO IREMOS CARREGAR AS TABELAS QUE ESTÃO NA SILVER PARA TRABALHAR COM ELAS


1.1 Criação da tabela gold.ft_vendas_consumidor_local

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.ft_vendas_consumidor_local AS 
SELECT 
  p.id_pedido, 
  p.id_consumidor, 
  CAST(p.valor_total_pago_brl AS DECIMAL(12,2)) AS valor_total_pago_brl, 
  CAST(p.data_pedido AS DATE) AS data_pedido, 
  c.cidade, 
  c.estado 
  FROM ecommerce.silver.ft_pedido_total p 
  LEFT JOIN ecommerce.silver.ft_consumidores c 
    ON p.id_consumidor = c.id_consumidor;

1.2 Criação da view
gold.view_total_compras_por_consumidor

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.view_total_compras_por_consumidor AS 
SELECT 
  cidade, 
  estado, 
  COUNT(*) AS quantidade_vendas, 
  SUM(valor_total_pago_brl) AS valor_total_localidade
FROM ecommerce.gold.ft_vendas_consumidor_local 
GROUP BY cidade, estado;


Respondendo perguntas da área de negócio:
Crie uma consulta SQL para exibir o total de vendas por estado

In [0]:
%sql
SELECT
    estado,
    SUM(valor_total_localidade) AS `total vendas por estado`
FROM ecommerce.gold.view_total_compras_por_consumidor
GROUP BY estado
ORDER BY `total vendas por estado` DESC;


2º Projeto — Área de Logística (Análise de Atrasos
de Entregas)

## 2.1 Criação da tabela
## gold.ft_atrasos_pedidos_local_vendedor:
## Cada linha representa um pedido com suas informações logísticas
### básicas.
### As informações devem ser obtidas a partir de:
### silver.ft_pedidos
### silver.ft_consumidores
### silver.ft_itens_pedidos

In [0]:
%sql
DESCRIBE ecommerce.silver.ft_pedidos;


In [0]:
%sql
DESCRIBE ecommerce.silver.ft_consumidores;


In [0]:
%sql
DESCRIBE ecommerce.silver.ft_itens_pedidos;


PRIMEIRO IREI DAR UM JOIN NA TABELA DE CONSUMIDORES, COM A DE PEDIDOS PARA DAR OUTRO JOIN NESSE RESULTADO

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.ft_atrasos_pedidos_local_vendedor
SELECT
  p.id_pedido,
  i.id_vendedor,
  p.id_consumidor,
  p.entregue_no_prazo,
  p.tempo_entrega_dias,
  p.tempo_entrega_estimado_dias,
  c.cidade,
  c.estado
  FROM ecommerce.silver.ft_pedidos p
  LEFT JOIN ecommerce.silver.ft_consumidores c
    ON p.id_consumidor = c.id_consumidor
  LEFT JOIN ecommerce.silver.ft_itens_pedidos i
    ON p.id_pedido = i.id_pedido


2.2 Criação das Views Analíticas

2.2.1 gold.view_tempo_medio_entrega_localidade

irei usar o spark para construir a view em si

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.view_tempo_medio_entrega_localidade AS
SELECT
    cidade,
    estado,
    ROUND(AVG(tempo_entrega_dias), 2) AS tempo_medio_entrega,
    ROUND(AVG(tempo_entrega_estimado_dias), 2) AS tempo_medio_estimado,
    CASE 
        WHEN MAX(
            CASE 
                WHEN tempo_entrega_dias > tempo_entrega_estimado_dias THEN 1
                ELSE 0
            END
        ) = 1 THEN 'SIM'
        ELSE 'NÃO'
    END AS entrega_maior_que_estimado
FROM ecommerce.gold.ft_atrasos_pedidos_local_vendedor
GROUP BY cidade, estado;


2.2.2 gold.view_vendedor_pontualidade


In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.view_vendedor_pontualidade AS
SELECT
    id_vendedor,
    COUNT(*) as total_pedidos,
    COUNT(CASE WHEN entregue_no_prazo = 'NÃO' THEN 1 END) as pedidos_entregues_no_prazo,
    ROUND(
    COUNT(CASE WHEN tempo_entrega_dias > tempo_entrega_estimado_dias THEN 1 END) 
    / COUNT(*) * 100
, 2) AS percentual_atraso
FROM ecommerce.gold.ft_atrasos_pedidos_local_vendedor
GROUP BY id_vendedor

3º Projeto — Área Comercial (Análises de Vendas
por Período)

3.1 Criação da Dimensão de Tempo — gold.dm_tempo
Será necessário criar uma dimensão, para que auxilie nas análises temporais
em diferentes granularidades (ano, trimestre, mês, semana, dia, etc.). Utilize as
funções explode e sequence para gerar os valores entre as datas de início e fim.
